# 01 — Data Preprocessing

**Project:** Predictive Analytics for Enterprise Streaming Acquisitions  
**Course:** CMPS 451 — Data Mining, Big Data & Analytics (Spring 2026)  
**Team:** 11

---

## Objectives
1. Load all 7 IMDb TSV datasets using PySpark
2. Clean and filter data (remove adult content, irrelevant title types, missing values)
3. Filter unreliable ratings (numVotes < 100) per instructor feedback
4. Join all tables into a unified dataset
5. Integrate the IEEE DataPort User Ratings dataset (Dataset.npy)
6. Save the preprocessed data for downstream analysis

In [20]:
# ── Imports & Environment Setup ──
import os
import sys
import numpy as np

# Fix Windows PySpark: set HADOOP_HOME and Python paths
os.environ['HADOOP_HOME'] = r'C:\hadoop'
os.environ['PATH'] = r'C:\hadoop\bin;' + os.environ.get('PATH', '')
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, FloatType
)

# Paths
BASE_DIR = r"e:\CUFE\Spring 25\Big Data\Project"
DATA_DIR = os.path.join(BASE_DIR, "Dataset")
OUTPUT_DIR = os.path.join(BASE_DIR, "outputs")
os.makedirs(os.path.join(OUTPUT_DIR, "parquet"), exist_ok=True)

print("Setup complete.")

Setup complete.


In [21]:
# ── Initialize Spark (pseudo-distributed mode) ──
spark = (
    SparkSession.builder
    .appName("IMDb_Preprocessing")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.ui.showConsoleProgress", "true")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")
print(f"Running in pseudo-distributed mode with {spark.sparkContext.defaultParallelism} cores")

Spark version: 4.1.1
Running in pseudo-distributed mode with 12 cores


## 1. Load IMDb Datasets

In [22]:
def load_tsv(filename):
    """Load a gzipped TSV file into a Spark DataFrame."""
    path = os.path.join(DATA_DIR, filename)
    df = (
        spark.read
        .option("header", "true")
        .option("sep", "\t")
        .option("nullValue", "\\N")
        .option("quote", "")
        .csv(path)
    )
    print(f"Loaded {filename}: {df.count():,} rows, {len(df.columns)} columns")
    return df

# Load all tables
df_basics    = load_tsv("title.basics.tsv.gz")
df_ratings   = load_tsv("title.ratings.tsv.gz")
df_crew      = load_tsv("title.crew.tsv.gz")
df_akas      = load_tsv("title.akas.tsv.gz")
df_principals= load_tsv("title.principals.tsv.gz")
df_episodes  = load_tsv("title.episode.tsv.gz")
df_names     = load_tsv("name.basics.tsv.gz")

Loaded title.basics.tsv.gz: 12,473,675 rows, 9 columns
Loaded title.ratings.tsv.gz: 1,667,592 rows, 3 columns
Loaded title.crew.tsv.gz: 12,473,675 rows, 3 columns
Loaded title.akas.tsv.gz: 56,610,555 rows, 8 columns
Loaded title.principals.tsv.gz: 99,234,353 rows, 6 columns
Loaded title.episode.tsv.gz: 9,633,850 rows, 4 columns
Loaded name.basics.tsv.gz: 15,290,041 rows, 6 columns


In [23]:
# ── Inspect schemas ──
print("=== title.basics ===")
df_basics.printSchema()
df_basics.show(3, truncate=False)

print("\n=== title.ratings ===")
df_ratings.printSchema()
df_ratings.show(3, truncate=False)

=== title.basics ===
root
 |-- tconst: string (nullable = true)
 |-- titleType: string (nullable = true)
 |-- primaryTitle: string (nullable = true)
 |-- originalTitle: string (nullable = true)
 |-- isAdult: string (nullable = true)
 |-- startYear: string (nullable = true)
 |-- endYear: string (nullable = true)
 |-- runtimeMinutes: string (nullable = true)
 |-- genres: string (nullable = true)

+---------+---------+----------------------+----------------------+-------+---------+-------+--------------+------------------------+
|tconst   |titleType|primaryTitle          |originalTitle         |isAdult|startYear|endYear|runtimeMinutes|genres                  |
+---------+---------+----------------------+----------------------+-------+---------+-------+--------------+------------------------+
|tt0000001|short    |Carmencita            |Carmencita            |0      |1894     |NULL   |1             |Documentary,Short       |
|tt0000002|short    |Le clown et ses chiens|Le clown et ses chiens

## 2. Data Cleaning & Filtering

In [24]:
# ── Step 2a: Filter by title type ──
VALID_TYPES = ["movie", "tvSeries", "tvMovie", "tvMiniSeries"]
df_basics_filtered = df_basics.filter(F.col("titleType").isin(VALID_TYPES))

print("Title type distribution (after filter):")
df_basics_filtered.groupBy("titleType").count().orderBy("count", ascending=False).show()

Title type distribution (after filter):
+------------+------+
|   titleType| count|
+------------+------+
|       movie|744928|
|    tvSeries|298742|
|     tvMovie|154755|
|tvMiniSeries| 69928|
+------------+------+



In [25]:
# ── Step 2b: Remove adult content ──
df_basics_filtered = df_basics_filtered.filter(F.col("isAdult") == "0")
print(f"After removing adult content: {df_basics_filtered.count():,} titles")

After removing adult content: 1,255,568 titles


In [26]:
# ── Step 2c: Cast numeric columns ──
df_basics_filtered = (
    df_basics_filtered
    .withColumn("startYear", F.col("startYear").cast(IntegerType()))
    .withColumn("endYear", F.col("endYear").cast(IntegerType()))
    .withColumn("runtimeMinutes", F.col("runtimeMinutes").cast(IntegerType()))
)

df_ratings = (
    df_ratings
    .withColumn("averageRating", F.col("averageRating").cast(FloatType()))
    .withColumn("numVotes", F.col("numVotes").cast(IntegerType()))
)

print("Numeric columns cast successfully.")

Numeric columns cast successfully.


In [27]:
# ── Step 2d: Drop rows with critical missing values ──
before = df_basics_filtered.count()
df_basics_clean = df_basics_filtered.dropna(subset=["startYear", "runtimeMinutes", "genres"])
after = df_basics_clean.count()
print(f"Dropped {before - after:,} rows with missing startYear/runtimeMinutes/genres")
print(f"Remaining: {after:,} titles")

Dropped 597,563 rows with missing startYear/runtimeMinutes/genres
Remaining: 658,005 titles


## 3. Join Tables

In [28]:
# ── Step 3a: Join basics with ratings ──
df_main = df_basics_clean.join(df_ratings, on="tconst", how="inner")
print(f"After joining with ratings: {df_main.count():,} titles")
df_main.show(3, truncate=False)

After joining with ratings: 415,104 titles
+---------+---------+-----------------------------+-----------------------------+-------+---------+-------+--------------+--------------------------+-------------+--------+
|tconst   |titleType|primaryTitle                 |originalTitle                |isAdult|startYear|endYear|runtimeMinutes|genres                    |averageRating|numVotes|
+---------+---------+-----------------------------+-----------------------------+-------+---------+-------+--------------+--------------------------+-------------+--------+
|tt0000009|movie    |Miss Jerry                   |Miss Jerry                   |0      |1894     |NULL   |45            |Romance                   |5.3          |240     |
|tt0000147|movie    |The Corbett-Fitzsimmons Fight|The Corbett-Fitzsimmons Fight|0      |1897     |NULL   |100           |Documentary,News,Sport    |5.3          |602     |
|tt0000574|movie    |The Story of the Kelly Gang  |The Story of the Kelly Gang  |0      |190

In [29]:
# ── Step 3b: Filter by numVotes (instructor feedback) ──
MIN_VOTES = 100
before = df_main.count()
df_main = df_main.filter(F.col("numVotes") >= MIN_VOTES)
after = df_main.count()
print(f"Filtered titles with numVotes < {MIN_VOTES}")
print(f"Removed: {before - after:,} titles")
print(f"Remaining: {after:,} titles with reliable ratings")

Filtered titles with numVotes < 100
Removed: 229,002 titles
Remaining: 186,102 titles with reliable ratings


In [30]:
# ── Step 3c: Join with crew (directors and writers) ──
df_main = df_main.join(df_crew, on="tconst", how="left")
print(f"After joining with crew: {df_main.count():,} titles")

After joining with crew: 186,102 titles


In [31]:
# ── Step 3d: Aggregate language info from title.akas ──
df_lang = (
    df_akas
    .filter(F.col("language").isNotNull())
    .groupBy("titleId", "language")
    .count()
    .withColumn(
        "rank",
        F.row_number().over(
            Window.partitionBy("titleId").orderBy(F.desc("count"))
        )
    )
    .filter(F.col("rank") == 1)
    .select(
        F.col("titleId").alias("tconst"),
        F.col("language").alias("primaryLanguage")
    )
)

df_region_count = (
    df_akas
    .filter(F.col("region").isNotNull())
    .groupBy("titleId")
    .agg(F.countDistinct("region").alias("numRegions"))
    .withColumnRenamed("titleId", "tconst")
)

df_main = df_main.join(df_lang, on="tconst", how="left")
df_main = df_main.join(df_region_count, on="tconst", how="left")
print(f"After joining with language/region info: {df_main.count():,} titles")

After joining with language/region info: 186,102 titles


In [32]:
# ── Step 3e: Aggregate principal cast/crew info ──
df_principals_agg = (
    df_principals
    .groupBy("tconst")
    .agg(
        F.count("nconst").alias("numPrincipals"),
        F.countDistinct("category").alias("numRoleTypes"),
        F.collect_set("nconst").alias("principalIds")
    )
)
df_main = df_main.join(df_principals_agg, on="tconst", how="left")
print(f"After joining with principals: {df_main.count():,} titles")

After joining with principals: 186,102 titles


## 4. Integrate User Ratings (Dataset.npy)

In [33]:
# ── Load the IEEE DataPort IMDb Users' Ratings Dataset ──
print("Loading Dataset.npy (4.67M user ratings)...")
user_ratings_raw = np.load(os.path.join(DATA_DIR, "Dataset.npy"), allow_pickle=True)
print(f"Loaded {len(user_ratings_raw):,} user ratings")
print(f"Sample: {user_ratings_raw[:3]}")

Loading Dataset.npy (4.67M user ratings)...
Loaded 4,669,820 user ratings
Sample: ['ur4592644,tt0120884,10,16 January 2005'
 'ur3174947,tt0118688,3,16 January 2005'
 'ur3780035,tt0387887,8,16 January 2005']


In [34]:
# ── Parse user ratings and load via CSV (avoids Python worker issues) ──
import pandas as pd

parsed = []
for row in user_ratings_raw:
    parts = row.split(",")
    if len(parts) >= 3:
        parsed.append((parts[0].strip(), parts[1].strip(), int(parts[2].strip())))

pdf_user = pd.DataFrame(parsed, columns=["userId", "titleId", "userRating"])
print(f"Parsed {len(pdf_user):,} ratings")
print(f"Unique users: {pdf_user['userId'].nunique():,}")
print(f"Unique titles: {pdf_user['titleId'].nunique():,}")

# Save to temp CSV then read with Spark's JVM CSV reader
# (avoids spark.createDataFrame which needs Python workers)
tmp_csv = os.path.join(OUTPUT_DIR, "_tmp_user_ratings.csv")
pdf_user.to_csv(tmp_csv, index=False)

user_schema = StructType([
    StructField("userId", StringType(), True),
    StructField("titleId", StringType(), True),
    StructField("userRating", IntegerType(), True)
])
df_user_ratings = spark.read.csv(tmp_csv, header=True, schema=user_schema)
df_user_ratings.show(5)
print(f"Spark DataFrame: {df_user_ratings.count():,} rows")

Parsed 4,669,820 ratings
Unique users: 1,499,238
Unique titles: 351,109
+------+-------+----------+
|userId|titleId|userRating|
+------+-------+----------+
+------+-------+----------+

Spark DataFrame: 0 rows


In [35]:
# ── Aggregate user-level features per title ──
df_user_agg = (
    df_user_ratings
    .groupBy("titleId")
    .agg(
        F.count("userRating").alias("userRatingCount"),
        F.mean("userRating").alias("userRatingMean"),
        F.stddev("userRating").alias("userRatingStd"),
        F.min("userRating").alias("userRatingMin"),
        F.max("userRating").alias("userRatingMax"),
        F.sum(F.when((F.col("userRating") <= 2) | (F.col("userRating") >= 9), 1).otherwise(0)).alias("extremeRatingCount")
    )
    .withColumn("extremeRatingRatio", F.col("extremeRatingCount") / F.col("userRatingCount"))
    .withColumn("userRatingRange", F.col("userRatingMax") - F.col("userRatingMin"))
    .withColumnRenamed("titleId", "tconst")
)

print("User rating aggregations:")
df_user_agg.show(5)

User rating aggregations:
+------+---------------+--------------+-------------+-------------+-------------+------------------+------------------+---------------+
|tconst|userRatingCount|userRatingMean|userRatingStd|userRatingMin|userRatingMax|extremeRatingCount|extremeRatingRatio|userRatingRange|
+------+---------------+--------------+-------------+-------------+-------------+------------------+------------------+---------------+
+------+---------------+--------------+-------------+-------------+-------------+------------------+------------------+---------------+



In [36]:
# ── Join user features with main dataset ──
df_main = df_main.join(df_user_agg, on="tconst", how="left")

user_cols = ["userRatingCount", "userRatingMean", "userRatingStd",
             "userRatingMin", "userRatingMax", "extremeRatingCount",
             "extremeRatingRatio", "userRatingRange"]
df_main = df_main.fillna(0, subset=user_cols)

print(f"Final dataset: {df_main.count():,} titles")
print(f"Columns: {len(df_main.columns)}")

Final dataset: 186,102 titles
Columns: 26


## 5. Summary & Save

In [37]:
# ── Summary statistics ──
print("=== Final Schema ===")
df_main.printSchema()

print("\n=== Numeric Summary ===")
df_main.select("averageRating", "numVotes", "runtimeMinutes", "startYear",
               "numRegions", "numPrincipals", "userRatingCount", "userRatingMean").describe().show()

print("\n=== Title Type Distribution ===")
df_main.groupBy("titleType").count().orderBy("count", ascending=False).show()

print("\n=== Language Distribution (Top 15) ===")
df_main.groupBy("primaryLanguage").count().orderBy("count", ascending=False).show(15)

=== Final Schema ===
root
 |-- tconst: string (nullable = true)
 |-- titleType: string (nullable = true)
 |-- primaryTitle: string (nullable = true)
 |-- originalTitle: string (nullable = true)
 |-- isAdult: string (nullable = true)
 |-- startYear: integer (nullable = true)
 |-- endYear: integer (nullable = true)
 |-- runtimeMinutes: integer (nullable = true)
 |-- genres: string (nullable = true)
 |-- averageRating: float (nullable = true)
 |-- numVotes: integer (nullable = true)
 |-- directors: string (nullable = true)
 |-- writers: string (nullable = true)
 |-- primaryLanguage: string (nullable = true)
 |-- numRegions: long (nullable = true)
 |-- numPrincipals: long (nullable = true)
 |-- numRoleTypes: long (nullable = true)
 |-- principalIds: array (nullable = true)
 |    |-- element: string (containsNull = false)
 |-- userRatingCount: long (nullable = true)
 |-- userRatingMean: double (nullable = false)
 |-- userRatingStd: double (nullable = false)
 |-- userRatingMin: integer (null

In [38]:
# ── Save preprocessed data ──
output_path = os.path.join(OUTPUT_DIR, "parquet", "preprocessed")
df_save = df_main.drop("principalIds")
df_save.write.mode("overwrite").parquet(output_path)

print(f"Preprocessed data saved to: {output_path}")
print(f"Total rows: {df_save.count():,}")
print(f"Total columns: {len(df_save.columns)}")

# Clean up temp CSV
if os.path.exists(tmp_csv):
    os.remove(tmp_csv)
    print("Temp CSV cleaned up.")

Py4JJavaError: An error occurred while calling o743.parquet.
: java.lang.RuntimeException: java.io.FileNotFoundException: java.io.FileNotFoundException: HADOOP_HOME and hadoop.home.dir are unset. -see https://cwiki.apache.org/confluence/display/HADOOP2/WindowsProblems
	at org.apache.hadoop.util.Shell.getWinUtilsPath(Shell.java:789)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:298)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:314)
	at org.apache.hadoop.fs.RawLocalFileSystem.setPermission(RawLocalFileSystem.java:1179)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkOneDirWithMode(RawLocalFileSystem.java:861)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:901)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirs(RawLocalFileSystem.java:873)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:900)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdir

In [ ]:
# ── Verify saved data ──
df_check = spark.read.parquet(output_path)
print(f"Verification - loaded {df_check.count():,} rows from parquet")
df_check.show(5, truncate=False)

In [ ]:
# Keep Spark session alive for next notebook (or stop here)
# spark.stop()
print("Preprocessing complete! Proceed to 02_feature_engineering.ipynb")